### **8. `MessagesPlaceholder` (for Inserting Chat History)**

This is a special placeholder within a `ChatPromptTemplate` used to inject a **list of messages** (the entire chat history) at a specific point in the prompt. This is essential for chatbots that need context from previous sessions.

*   **Scenario:** A customer had a chat yesterday. Today, they ask "Where is my refund?". You need to inject yesterday's conversation as context.

In [1]:
import json
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

### function to save chat

In [ ]:
## Assume this is the history loaded from a database/file
def save_chat(chat_history, filename="./1.2MessagePlaceHolder_chat_history.json"):
    data = []
    for msg in chat_history:
        # Skip SystemMessage
        if isinstance(msg, SystemMessage):
            continue
        
        data.append({
            "type": msg.__class__.__name__,
            "content": msg.content
        })

    with open(filename, "w") as f:
        json.dump(data, f, indent=2)

### Function to load chat

In [3]:
def load_chat(filename="./1.2MessagePlaceHolder_chat_history.json"):
    messages = []
    with open(filename) as f:
        data = json.load(f)
    for msg in data:
        if msg["type"] == "HumanMessage":
            messages.append(HumanMessage(content=msg["content"]))
        elif msg["type"] == "AIMessage":
            messages.append(AIMessage(content=msg["content"]))
    return messages


## Example Chat_history variable

In [ ]:
 # Assume this is the history loaded from a database/file
chat_history = [
    SystemMessage(content="You are a helpful assistant"),
    HumanMessage(content="Hi"),
    AIMessage(content="Hello! I'm here to help."),
    HumanMessage(content="Who is Narendra Modi?"),
    AIMessage(content="Narendra Modi is the Prime Minister of India."),
    HumanMessage(content="How old is he?"),
    AIMessage(content="He was born on September 17, 1950."),
]

In [ ]:
# Save chat (system messages skipped) using those defined functions above
save_chat(chat_history)

# Load chat back
chat_history = load_chat()

In [ ]:
# 1.Create chatprompt template with a  MessagePlaceHolder
chat_template = ChatPromptTemplate(
    [
        ("system", "You are a helpful customer support agent"),
        MessagesPlaceholder(variable_name="chat_history"), #History will be inserted here
        ("human", "{query}"), #Current query
    ]
)

In [ ]:
# 2.Format the template by providing the history and current input
query="Modi's wife name is?"
prompt=chat_template.invoke(
    {
        "chat_history":chat_history,
        "query":query
    }
)

In [ ]:
print(prompt) #Now when this prompt is sent to the LLM ,it will have full context.

messages=[SystemMessage(content='You are a helpful customer support agent', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hi', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello! I'm here to help.", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='Who is Narendra Modi?', additional_kwargs={}, response_metadata={}), AIMessage(content='Narendra Modi is the Prime Minister of India.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='How old is he?', additional_kwargs={}, response_metadata={}), AIMessage(content='He was born on September 17, 1950.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content="Modi's wife name is?", additional_kwargs={}, response_metadata={})]


## If you want to   pass it to the model

In [10]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

api_key = os.getenv("Hugging_face_api_token")

# Create LLM endpoint
llm = HuggingFaceEndpoint(
    # repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    repo_id="mistralai/Mistral-7B-Instruct-v0.2",
    task="text-generation",
    huggingfacehub_api_token=api_key,
)

# Wrap with chat interface
model = ChatHuggingFace(llm=llm)

In [ ]:
result=model.invoke(prompt)
print(result.content)

 Prime Minister Narendra Modi is not married. He has lived a celibate life since he was a young man and has no family. He has often spoken about his devotion to public service and being fully committed to the people of India.
